# Forecasting

Carga de los datos de inferencia para 2025.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import sklearn.dummy
import streamlit as st
import holidays

inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_df.head()

## Preparacion de `inferencia_df`

Se reproducen las transformaciones aplicadas en el notebook de entrenamiento y se conserva octubre durante el calculo de retardos. Al final solo se guardan los registros de noviembre de 2025.

In [ ]:
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'], errors='coerce')

fecha_valida = inferencia_df['fecha'].notna()
años_disponibles = inferencia_df.loc[fecha_valida, 'fecha'].dt.year

if años_disponibles.empty:
    raise ValueError('No hay fechas validas para crear variables temporales.')

calendario_espana = holidays.Spain(
    years=range(int(años_disponibles.min()), int(años_disponibles.max()) + 1)
)
nombres_festivos = {
    pd.Timestamp(fecha): nombre
    for fecha, nombre in calendario_espana.items()
}

black_friday_por_año = {}
cyber_monday_por_año = {}
for año in range(int(años_disponibles.min()), int(años_disponibles.max()) + 1):
    primer_dia_noviembre = pd.Timestamp(year=año, month=11, day=1)
    dias_hasta_primer_jueves = (3 - primer_dia_noviembre.weekday()) % 7
    cuarto_jueves = primer_dia_noviembre + pd.Timedelta(
        days=dias_hasta_primer_jueves + 21
    )
    black_friday_por_año[año] = cuarto_jueves + pd.Timedelta(days=1)
    cyber_monday_por_año[año] = cuarto_jueves + pd.Timedelta(days=4)

fecha_normalizada = inferencia_df['fecha'].dt.normalize()
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week.astype('Int64')
inferencia_df['dia_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['dia_semana_numero'] = inferencia_df['fecha'].dt.weekday
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana_numero'].isin([5, 6])
inferencia_df['es_inicio_semana'] = inferencia_df['dia_semana_numero'].eq(0)
inferencia_df['es_viernes'] = inferencia_df['dia_semana_numero'].eq(4)
inferencia_df['es_inicio_mes'] = inferencia_df['dia_mes'].eq(1)
inferencia_df['es_fin_mes'] = inferencia_df['fecha'].dt.is_month_end
inferencia_df['es_inicio_trimestre'] = inferencia_df['fecha'].dt.is_quarter_start
inferencia_df['es_fin_trimestre'] = inferencia_df['fecha'].dt.is_quarter_end
inferencia_df['es_festivo'] = (
    fecha_normalizada.dt.date.map(calendario_espana)
    .fillna(False)
    .astype(bool)
)
inferencia_df['nombre_festivo'] = (
    fecha_normalizada.map(nombres_festivos).fillna('No festivo')
)
inferencia_df['es_black_friday'] = inferencia_df['fecha'].map(
    black_friday_por_año
).eq(fecha_normalizada)
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].map(
    cyber_monday_por_año
).eq(fecha_normalizada)
inferencia_df['es_navidad'] = (
    inferencia_df['mes'].eq(12) & inferencia_df['dia_mes'].eq(25)
)
inferencia_df['es_nochebuena'] = (
    inferencia_df['mes'].eq(12) & inferencia_df['dia_mes'].eq(24)
)
inferencia_df['es_reyes'] = (
    inferencia_df['mes'].eq(1) & inferencia_df['dia_mes'].eq(6)
)
inferencia_df['dias_desde_inicio_año'] = inferencia_df['fecha'].dt.dayofyear - 1

inferencia_df = inferencia_df.sort_values(
    ['producto_id', 'año', 'fecha']
).copy()
grupo_producto_año = inferencia_df.groupby(
    ['producto_id', 'año'],
    sort=False
)['unidades_vendidas']

variables_retardo = []
for numero_lag in range(1, 8):
    nombre_lag = f'unidades_vendidas_lag_{numero_lag}'
    inferencia_df[nombre_lag] = grupo_producto_año.shift(numero_lag)
    variables_retardo.append(nombre_lag)

inferencia_df['media_movil_7_dias'] = (
    grupo_producto_año.shift(1)
    .groupby([inferencia_df['producto_id'], inferencia_df['año']])
    .rolling(window=7, min_periods=7)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)
variables_retardo.append('media_movil_7_dias')
inferencia_df = inferencia_df.dropna(
    subset=variables_retardo
).reset_index(drop=True)

inferencia_df['descuento_porcentaje'] = np.where(
    inferencia_df['precio_base'].ne(0),
    (
        (inferencia_df['precio_venta'] - inferencia_df['precio_base'])
        / inferencia_df['precio_base']
    ) * 100,
    np.nan
)

columnas_competidores = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['precio_competencia'] = inferencia_df[
    columnas_competidores
].mean(axis=1)
inferencia_df['ratio_precio'] = np.where(
    inferencia_df['precio_competencia'].ne(0),
    inferencia_df['precio_venta'] / inferencia_df['precio_competencia'],
    np.nan
)
inferencia_df = inferencia_df.drop(columns=columnas_competidores)

variables_categoricas = ['nombre', 'categoria', 'subcategoria']
variables_categoricas_h = [f'{variable}_h' for variable in variables_categoricas]
for variable, variable_h in zip(variables_categoricas, variables_categoricas_h):
    inferencia_df[variable_h] = inferencia_df[variable]

inferencia_df = pd.get_dummies(
    inferencia_df,
    columns=variables_categoricas_h,
    prefix=variables_categoricas_h,
    dtype=int
)

print(
    f'Filas tras crear retardos y media movil: '
    f'{len(inferencia_df)}'
)
print(f'Columnas tras la ingenieria de variables: {inferencia_df.shape[1]}')

In [ ]:
inferencia_df.shape

In [ ]:
inferencia_df.columns
